# ETL Silver - Curvas de aforo y caudal diario (nivel -> caudal)

Corazon de la Fase 2 de `docs/rating_curve_discharge_plan.md`. Dos pasos:
1. Tipa y consolida `weather.bronze.ana_rating_curve_segments` / `ana_discharge_measurements`
   en `weather.silver.rating_curve_segments` (incluye QC contra aforos reales, D3).
2. Convierte el nivel diario de TODAS las estaciones con curva (leido directo de
   `weather.bronze.ana_rio_uruguai`, no de `river_levels_daily` que es solo la estacion
   target) en `weather.silver.river_discharge_daily`, eligiendo por fecha la curva
   vigente y extrapolando con flag en vez de anular (Decision D3).

In [ ]:
from datetime import date, timedelta

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BRONZE_CURVE_TABLE = 'weather.bronze.ana_rating_curve_segments'
BRONZE_DISCHARGE_TABLE = 'weather.bronze.ana_discharge_measurements'
BRONZE_LEVEL_TABLE = 'weather.bronze.ana_rio_uruguai'
SEGMENTS_TABLE = 'weather.silver.rating_curve_segments'
DISCHARGE_TABLE = 'weather.silver.river_discharge_daily'
QUALITY_TABLE = 'weather.silver.attribute_quality'

MAPE_USABLE_THRESHOLD = 0.20
DATASET_FLOOR = date(2000, 1, 1)  # Decision D4: nada anterior a esta fecha entra al dataset

try:
    dbutils.widgets.dropdown('load_mode', 'incremental', ['full', 'incremental'])
    dbutils.widgets.text('incremental_lookback_days', '14')
    load_mode = dbutils.widgets.get('load_mode')
    incremental_lookback_days = int(dbutils.widgets.get('incremental_lookback_days'))
except Exception:
    load_mode = 'incremental'
    incremental_lookback_days = 14

print(f'load_mode={load_mode}, incremental_lookback_days={incremental_lookback_days}')

In [ ]:
def parse_decimal(column_name):
    return F.regexp_replace(F.trim(F.col(column_name).cast('string')), ',', '.').cast('double')


def apply_incremental_window(df, target_table, date_col='fecha'):
    if load_mode == 'full':
        return df
    max_target_fecha = spark.table(target_table).agg(F.max(date_col).alias('max_fecha')).first()['max_fecha']
    if max_target_fecha is None:
        return df
    start_date = max_target_fecha - timedelta(days=incremental_lookback_days)
    print(f'Processing {target_table} from {start_date}')
    return df.filter(F.col(date_col) >= F.lit(start_date))


def build_quality(df, attribute_name, notes):
    return (
        df.agg(
            F.min('fecha').alias('evaluation_start_date'),
            F.max('fecha').alias('evaluation_end_date'),
            F.countDistinct(F.when(F.col(attribute_name).isNotNull(), F.col('fecha'))).cast('bigint').alias('observed_days'),
        )
        .withColumn('expected_days', F.when(F.col('evaluation_start_date').isNull(), F.lit(0)).otherwise(F.datediff(F.col('evaluation_end_date'), F.col('evaluation_start_date')) + F.lit(1)).cast('bigint'))
        .withColumn('missing_days', F.greatest(F.col('expected_days') - F.col('observed_days'), F.lit(0)).cast('bigint'))
        .withColumn('missing_pct', F.when(F.col('expected_days') == 0, F.lit(1.0)).otherwise(F.col('missing_days') / F.col('expected_days')))
        .withColumn('threshold_pct', F.lit(0.90))
        .withColumn('is_usable', F.col('missing_pct') <= F.col('threshold_pct'))
        .withColumn('source_layer', F.lit('silver'))
        .withColumn('source_table', F.lit(DISCHARGE_TABLE))
        .withColumn('source_name', F.lit('river_discharge_daily'))
        .withColumn('attribute_name', F.lit(attribute_name))
        .withColumn('grain', F.lit('global_source_daily'))
        .withColumn('evaluated_at', F.current_timestamp())
        .withColumn('notes', F.lit(notes))
        .withColumn('created_at', F.current_timestamp())
        .withColumn('updated_at', F.current_timestamp())
        .select('source_layer', 'source_table', 'source_name', 'attribute_name', 'grain', 'evaluation_start_date', 'evaluation_end_date', 'expected_days', 'observed_days', 'missing_days', 'missing_pct', 'threshold_pct', 'is_usable', 'evaluated_at', 'notes', 'created_at', 'updated_at')
    )


def merge_quality(quality_df):
    DeltaTable.forName(spark, QUALITY_TABLE).alias('t').merge(
        quality_df.alias('s'),
        't.source_table = s.source_table AND t.attribute_name = s.attribute_name AND t.grain = s.grain',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def merge_segments(df):
    if df.limit(1).count() == 0:
        print('No hay segmentos para mergear')
        return
    DeltaTable.forName(spark, SEGMENTS_TABLE).alias('t').merge(
        df.alias('s'),
        't.codigoestacao = s.codigoestacao AND t.rating_curve_id = s.rating_curve_id AND t.segment_number = s.segment_number',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


def merge_discharge(df):
    if df.limit(1).count() == 0:
        print('No hay filas de caudal para mergear')
        return
    DeltaTable.forName(spark, DISCHARGE_TABLE).alias('t').merge(
        df.alias('s'),
        't.fecha = s.fecha AND t.codigoestacao = s.codigoestacao',
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Paso 1: tipar y consolidar `weather.silver.rating_curve_segments`

Siempre en modo `full` sobre los segmentos (son ~510 filas para el grupo A y unas pocas
miles en el universo completo -- no hace falta incrementalidad, ver plan Seccion 4.2).

In [ ]:
raw_segments = spark.table(BRONZE_CURVE_TABLE)

typed_segments = (
    raw_segments
    .withColumn('codigoestacao', F.col('codigoestacao').cast('string'))
    .withColumn('valid_from', F.to_date('Periodo_Validade_Inicio'))
    .withColumn('valid_to', F.to_date('Periodo_Validade_Fim'))
    .withColumn('segment_number', F.split(F.col('Numero_Curva'), '/').getItem(0).cast('int'))
    .withColumn('coefficient_a', parse_decimal('Coef_a'))
    .withColumn('coefficient_h0_m', parse_decimal('Coef_h0'))
    .withColumn('coefficient_n', parse_decimal('Coef_n'))
    .withColumn('stage_min_cm', parse_decimal('Cota_Minima'))
    .withColumn('stage_max_cm', parse_decimal('Cota_Maxima'))
    .withColumn('consistency_level', F.col('Nivel_Consistencia').cast('int'))
    .withColumn(
        'rating_curve_id',
        F.concat_ws('_', F.col('codigoestacao'), F.col('Periodo_Validade_Inicio'), F.col('Periodo_Validade_Fim')),
    )
    .filter(F.col('valid_from').isNotNull() & F.col('coefficient_a').isNotNull() & F.col('coefficient_h0_m').isNotNull() & F.col('coefficient_n').isNotNull())
    .filter(F.col('stage_min_cm').isNotNull() & F.col('stage_max_cm').isNotNull())
)

# Ante duplicados exactos (mismo codigoestacao+rating_curve_id+segment_number llegado en
# mas de una ventana de descarga), preferir mayor consistency_level (2=consistido > 1=bruto).
dedup_window = Window.partitionBy('codigoestacao', 'rating_curve_id', 'segment_number').orderBy(F.col('consistency_level').desc_nulls_last())
typed_segments = (
    typed_segments
    .withColumn('_rn', F.row_number().over(dedup_window))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)

curve_window = Window.partitionBy('codigoestacao', 'rating_curve_id')
typed_segments = (
    typed_segments
    .withColumn('_curve_min', F.min('stage_min_cm').over(curve_window))
    .withColumn('_curve_max', F.max('stage_max_cm').over(curve_window))
    .withColumn('is_lowest_segment', F.col('stage_min_cm') == F.col('_curve_min'))
    .withColumn('is_highest_segment', F.col('stage_max_cm') == F.col('_curve_max'))
    .drop('_curve_min', '_curve_max')
)

print(f'{typed_segments.count()} segmentos tipados, {typed_segments.select("codigoestacao").distinct().count()} estaciones')

In [ ]:
# QC: aforo maximo real por estacion (limite empirico, D3) + MAPE contra aforos que caen
# dentro del rango calibrado del segmento vigente en su fecha (misma logica que
# evaluate_curve_accuracy() en notebooks_local/ana_rating_curve/download_rating_curve.py,
# reescrita en Spark).
raw_measurements = spark.table(BRONZE_DISCHARGE_TABLE)

typed_measurements = (
    raw_measurements
    .withColumn('codigoestacao', F.col('codigoestacao').cast('string'))
    .withColumn('medicao_ts', F.to_timestamp('Data_Hora_Dado'))
    .withColumn('stage_cm', parse_decimal('Cota'))
    .withColumn('vazao_real', parse_decimal('Vazao'))
    .filter(F.col('stage_cm').isNotNull() & F.col('vazao_real').isNotNull() & (F.col('vazao_real') > 0))
)

aforo_stage_max = typed_measurements.groupBy('codigoestacao').agg(F.max('stage_cm').alias('aforo_stage_max_cm'))

matched_aforos = (
    typed_measurements.alias('a')
    .join(
        typed_segments.alias('c'),
        (F.col('a.codigoestacao') == F.col('c.codigoestacao'))
        & (F.col('a.medicao_ts') >= F.col('c.valid_from'))
        & (F.col('a.medicao_ts') <= F.col('c.valid_to'))
        & (F.col('a.stage_cm') >= F.col('c.stage_min_cm'))
        & (F.col('a.stage_cm') <= F.col('c.stage_max_cm')),
        'inner',
    )
    .withColumn('delta_m', (F.col('a.stage_cm') / F.lit(100.0)) - F.col('c.coefficient_h0_m'))
    .filter(F.col('delta_m') > 0)
    .withColumn('caudal_predicho', F.col('c.coefficient_a') * F.pow(F.col('delta_m'), F.col('c.coefficient_n')))
    .withColumn('error_pct', F.abs(F.col('caudal_predicho') - F.col('a.vazao_real')) / F.col('a.vazao_real'))
)

station_mape = matched_aforos.groupBy(F.col('a.codigoestacao').alias('codigoestacao')).agg(
    F.avg('error_pct').alias('validation_mape'),
    F.count('*').alias('n_aforos_matched'),
)

typed_segments = (
    typed_segments
    .join(aforo_stage_max, on='codigoestacao', how='left')
    .join(station_mape, on='codigoestacao', how='left')
    .withColumn('is_usable', F.col('validation_mape').isNull() | (F.col('validation_mape') <= F.lit(MAPE_USABLE_THRESHOLD)))
    .withColumn('source_table', F.lit(BRONZE_CURVE_TABLE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select(
        'codigoestacao', 'rating_curve_id', 'segment_number', 'valid_from', 'valid_to',
        'stage_min_cm', 'stage_max_cm', 'coefficient_a', 'coefficient_h0_m', 'coefficient_n',
        'consistency_level', 'is_lowest_segment', 'is_highest_segment', 'aforo_stage_max_cm',
        'is_usable', 'validation_mape', 'source_table', 'processed_at', 'updated_at',
    )
)

merge_segments(typed_segments)
print('rating_curve_segments actualizado')

## Paso 2: nivel diario (todas las estaciones con curva) -> caudal diario

Se lee `weather.bronze.ana_rio_uruguai` en vez de `weather.silver.river_levels_daily`
porque esa tabla Silver solo cubre la estacion target (74100000); aca hace falta el
nivel diario de todas las estaciones que tengan curva.

In [ ]:
segments_current = spark.table(SEGMENTS_TABLE)
estaciones_con_curva = [r['codigoestacao'] for r in segments_current.select('codigoestacao').distinct().collect()]
print(f'{len(estaciones_con_curva)} estaciones con curva disponible')

nivel_diario = (
    spark.table(BRONZE_LEVEL_TABLE)
    .select('codigoestacao', 'Data_Hora_Medicao', 'Cota_Adotada')
    .withColumn('codigoestacao', F.col('codigoestacao').cast('string'))
    .filter(F.col('codigoestacao').isin(estaciones_con_curva))
    .withColumn('medicao_ts', F.to_timestamp('Data_Hora_Medicao'))
    .withColumn('fecha', F.to_date('medicao_ts'))
    .withColumn('nivel_cm', parse_decimal('Cota_Adotada'))
    .filter(F.col('fecha').isNotNull() & F.col('nivel_cm').isNotNull())
    .filter(F.col('fecha') >= F.lit(DATASET_FLOOR))
    .groupBy('fecha', 'codigoestacao')
    .agg(F.avg('nivel_cm').alias('nivel_media_cm'))
    .withColumn('nivel_media_m', F.col('nivel_media_cm') / F.lit(100.0))
)

nivel_diario = apply_incremental_window(nivel_diario, DISCHARGE_TABLE)
print(f'{nivel_diario.count()} filas fecha x estacion a procesar')

In [ ]:
# Join temporal: para cada (fecha, codigoestacao), todos los segmentos de la vigencia
# activa en esa fecha (varias filas por match, una por segmento de esa curva).
candidatos = (
    nivel_diario.alias('n')
    .join(
        segments_current.alias('c'),
        (F.col('n.codigoestacao') == F.col('c.codigoestacao'))
        & (F.col('n.fecha') >= F.col('c.valid_from'))
        & (F.col('n.fecha') <= F.col('c.valid_to')),
        'left',
    )
)

group_window = Window.partitionBy('n.fecha', 'n.codigoestacao')
candidatos = (
    candidatos
    .withColumn('_group_min', F.min('c.stage_min_cm').over(group_window))
    .withColumn('_group_max', F.max('c.stage_max_cm').over(group_window))
    .withColumn(
        '_match_priority',
        F.when(F.col('c.rating_curve_id').isNull(), F.lit(9))  # sin vigencia para esa fecha
        .when((F.col('n.nivel_media_cm') >= F.col('c.stage_min_cm')) & (F.col('n.nivel_media_cm') <= F.col('c.stage_max_cm')), F.lit(1))
        .when(F.col('c.is_highest_segment') & (F.col('n.nivel_media_cm') > F.col('_group_max')), F.lit(2))
        .when(F.col('c.is_lowest_segment') & (F.col('n.nivel_media_cm') < F.col('_group_min')), F.lit(3))
        .otherwise(F.lit(9)),
    )
)

pick_window = Window.partitionBy('n.fecha', 'n.codigoestacao').orderBy('_match_priority')
picked = (
    candidatos
    .withColumn('_rn', F.row_number().over(pick_window))
    .filter(F.col('_rn') == 1)
    .drop('_rn')
)

print(f'{picked.count()} filas tras seleccionar segmento (debe igualar a las {nivel_diario.count()} de nivel_diario)')

In [ ]:
# Formula Q = A * (H/100 - H0)**N, convencion validada en docs/rating_curve_discharge_plan.md
# Seccion 2.2. Los 5 casos de la tabla de decision (D3) en Seccion 4.4:
delta_m = (F.col('n.nivel_media_cm') / F.lit(100.0)) - F.col('c.coefficient_h0_m')

resultado = (
    picked
    .withColumn('curva_disponible', F.col('c.rating_curve_id').isNotNull())
    .withColumn(
        'caudal_metodo',
        F.when(~F.col('curva_disponible'), F.lit('sin_curva'))
        .when(F.col('_match_priority') == 1, F.lit('interpolado'))
        .when(F.col('_match_priority') == 2, F.lit('extrapolado_superior'))
        .when((F.col('_match_priority') == 3) & (delta_m > 0), F.lit('extrapolado_inferior'))
        .when(F.col('_match_priority') == 3, F.lit('bajo_cero_curva'))
        .otherwise(F.lit('sin_curva')),
    )
    .withColumn(
        'caudal_m3s',
        F.when(F.col('caudal_metodo') == 'sin_curva', F.lit(None).cast('double'))
        .when(F.col('caudal_metodo') == 'bajo_cero_curva', F.lit(0.0))
        .when(delta_m > 0, F.col('c.coefficient_a') * F.pow(delta_m, F.col('c.coefficient_n')))
        .otherwise(F.lit(None).cast('double')),
    )
    .withColumn(
        'distancia_fuera_rango_cm',
        F.when(F.col('caudal_metodo') == 'extrapolado_superior', F.col('n.nivel_media_cm') - F.col('_group_max'))
        .when(F.col('caudal_metodo').isin('extrapolado_inferior', 'bajo_cero_curva'), F.col('_group_min') - F.col('n.nivel_media_cm'))
        .otherwise(F.lit(0.0)),
    )
    .withColumn(
        'supera_aforo_maximo',
        F.when(F.col('c.aforo_stage_max_cm').isNotNull(), F.col('n.nivel_media_cm') > F.col('c.aforo_stage_max_cm')).otherwise(F.lit(False)),
    )
    .withColumn('caudal_extrapolado', F.col('caudal_metodo').isin('extrapolado_superior', 'extrapolado_inferior'))
    .withColumn('caudal_confiable', (F.col('caudal_metodo') == 'interpolado') & F.coalesce(F.col('c.is_usable'), F.lit(True)))
    .withColumn('source_table', F.lit(BRONZE_LEVEL_TABLE))
    .withColumn('processed_at', F.current_timestamp())
    .withColumn('updated_at', F.current_timestamp())
    .select(
        F.col('n.fecha').alias('fecha'),
        F.col('n.codigoestacao').alias('codigoestacao'),
        F.col('n.nivel_media_cm').alias('nivel_media_cm'),
        F.col('n.nivel_media_m').alias('nivel_media_m'),
        'caudal_m3s',
        F.col('c.rating_curve_id').alias('rating_curve_id'),
        F.col('c.segment_number').alias('segment_number'),
        'caudal_metodo', 'caudal_extrapolado', 'distancia_fuera_rango_cm', 'supera_aforo_maximo',
        'curva_disponible', 'caudal_confiable', 'source_table', 'processed_at', 'updated_at',
    )
)

merge_discharge(resultado)
print('river_discharge_daily actualizado')
resultado.groupBy('caudal_metodo').count().show()

In [ ]:
full_discharge_for_quality = spark.table(DISCHARGE_TABLE)
merge_quality(build_quality(full_discharge_for_quality, 'caudal_m3s', 'Caudal diario calculado (todas las estaciones con curva)'))

spark.table(DISCHARGE_TABLE).agg(
    F.min('fecha').alias('inicio'), F.max('fecha').alias('fin'),
    F.count('*').alias('rows'), F.countDistinct('codigoestacao').alias('estaciones'),
).show()